# CKD Capstone: classification and interpretability

**Tan Yann Bin | Sunway University**

This notebook runs the repository reconstruction of the documented capstone. The original final notebook was unavailable. These results are generated from the included code; the submitted results are preserved separately in [docs/reported_results.md](docs/reported_results.md).

Install `requirements-notebook.txt` in a Python 3.12 environment and launch Jupyter from the repository directory.

## 1. Load and inspect the data

The target is **CKD = 1** and **non-CKD = 0**. Fixed text cleanup does not learn statistics. ID and target are excluded from the 24 predictors.

In [1]:
from pathlib import Path
import json
import tempfile
import pandas as pd
from ckd_pipeline import DEFAULT_DATA, load_dataset, split_dataset, run

X, y = load_dataset()
print(f"Records: {len(X)} | Predictors: {X.shape[1]}")
print("Target counts:", y.value_counts().sort_index().to_dict())
X.head()

Records: 400 | Predictors: 24
Target counts: {0: 150, 1: 250}
Out[1]: 
    age    bp     sg   al   su    bgr  ...  htn   dm  cad  appet   pe  ane
0  48.0  80.0  1.020  1.0  0.0  121.0  ...  yes  yes   no   good   no   no
1   7.0  50.0  1.020  4.0  0.0    NaN  ...   no   no   no   good   no   no
2  62.0  80.0  1.010  2.0  3.0  423.0  ...   no  yes   no   poor   no  yes
3  48.0  70.0  1.005  4.0  0.0  117.0  ...  yes   no   no   poor  yes  yes
4  51.0  80.0  1.010  2.0  0.0  106.0  ...   no   no   no   good   no   no

[5 rows x 24 columns]


## 2. Check the split

A stratified 80/20 split with seed 42 is used. All six models evaluate the same test records. Every imputer and scaler is fitted inside a model pipeline on training records only.

In [2]:
X_train, X_test, y_train, y_test = split_dataset(X, y, seed=42, test_size=0.2)
assert set(X_train.index).isdisjoint(X_test.index)
print(f"Training: {len(X_train)} | Test: {len(X_test)}")
print("Test target counts:", y_test.value_counts().sort_index().to_dict())

Training: 320 | Test: 80
Test target counts: {0: 30, 1: 50}


## 3. Train, evaluate and explain all six models

The same `run` function is used by the command-line script. It generates EDA from training data, metrics, confusion matrices, ROC curves, feature importance, and Random Forest SHAP explanations. No test-based tuning is performed. Each execution creates a fresh directory.

In [3]:
Path("outputs").mkdir(exist_ok=True)
run_dir = Path(tempfile.mkdtemp(prefix="notebook-", dir="outputs"))
metrics = run(DEFAULT_DATA, run_dir, seed=42, test_size=0.2, with_shap=True)
print("Saved results to:", run_dir)
metrics.round(4)

Saved results to: /workspace/scratch/cd6d10df5630/CKD-Capstone-Project/outputs/notebook-njgy7nu5
Out[3]: 
                    model  accuracy  precision  recall  ...  tn  fp  fn  tp
0     Logistic Regression    0.9875       1.00    0.98  ...  30   0   1  49
1           Decision Tree    0.9750       0.98    0.98  ...  29   1   1  49
2           Random Forest    1.0000       1.00    1.00  ...  30   0   0  50
3                 XGBoost    0.9875       1.00    0.98  ...  30   0   1  49
4   LR + RF (soft voting)    1.0000       1.00    1.00  ...  30   0   0  50
5  DT + XGB (soft voting)    0.9750       0.98    0.98  ...  29   1   1  49

[6 rows x 10 columns]


## 4. Inspect explanations and completed-run metadata

SHAP describes contributions to the Random Forest probability of CKD. Its background distribution comes from training path counts stored in the forest. The run verifies that baseline plus contributions equals predicted probability. These are model associations, not causal effects.

In [4]:
metadata = json.loads((run_dir / "run_metadata.json").read_text())
print("Run status:", metadata["status"])
print("SHAP additivity check:", metadata["shap"]["additivity_check"])
pd.read_csv(run_dir / "shap_importance.csv").head(10)

Run status: completed
SHAP additivity check: passed
Out[4]: 
  feature  mean_absolute_shap
0    hemo            0.092407
1      sg            0.079835
2     pcv            0.072871
3      sc            0.067781
4      rc            0.046713
5     htn            0.036402
6      al            0.035100
7      dm            0.031721
8     sod            0.016523
9     bgr            0.015646


## 5. View generated figures

Open the SVG files in the printed output folder, or view the saved examples below. The examples come from the recorded default run in `results/`.

![Confusion matrices](results/figures/confusion_matrices.svg)

![SHAP importance](results/figures/shap_importance.svg)

## Limitations

These results describe one 80-record holdout from a 400-record academic dataset. They do not establish clinical validity or generalisation to other hospitals. The target is existing CKD status, not future onset. Compare current metrics with the separately labelled original reported results; exact reproduction of the unavailable final notebook is not claimed.